# Anime Acquisition Strategy — Data Engineering
ITDS 2373 | Tobechukwu

# Import Libraries

In [1]:
import re 
import ast
import html
import pandas as pd 
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import nltk
from nltk.corpus import stopwords

# Load all Datasets

In [3]:
animes = pd.read_csv("animes.csv")
profiles = pd.read_csv("profiles.csv")
reviews = pd.read_csv("reviews.csv")

# Data Description and Relationships
#### Inspect the source datasets, their schemas, and potential join keys before cleaning.

In [4]:
datasets = {
    'Animes': animes,
    'Profiles': profiles,
    'Reviews': reviews
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name + ' Dataset':^40}")
    print(f"{'='*40}")

    print("\nFirst 5 Rows:")
    display(df.head())
    
    print("\nDataset Information:")
    df.info()

    print("\nDescriptive Statistics:")
    display(df.describe(include='all'))

    #print("\nShape:", df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    


             Animes Dataset             

First 5 Rows:


,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,https://cdn.myanimelist.net/images/anime/9/766...,https://myanimelist.net/anime/28891/Haikyuu_Se...
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,https://cdn.myanimelist.net/images/anime/3/671...,https://myanimelist.net/anime/23273/Shigatsu_w...
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,https://cdn.myanimelist.net/images/anime/6/867...,https://myanimelist.net/anime/34599/Made_in_Abyss
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,https://cdn.myanimelist.net/images/anime/1223/...,https://myanimelist.net/anime/5114/Fullmetal_A...
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,https://cdn.myanimelist.net/images/anime/3/815...,https://myanimelist.net/anime/31758/Kizumonoga...



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19311 entries, 0 to 19310
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   uid         19311 non-null  int64  
 1   title       19311 non-null  object 
 2   synopsis    18336 non-null  object 
 3   genre       19311 non-null  object 
 4   aired       19311 non-null  object 
 5   episodes    18605 non-null  float64
 6   members     19311 non-null  int64  
 7   popularity  19311 non-null  int64  
 8   ranked      16099 non-null  float64
 9   score       18732 non-null  float64
 10  img_url     19131 non-null  object 
 11  link        19311 non-null  object 
dtypes: float64(3), int64(3), object(6)
memory usage: 1.8+ MB

Descriptive Statistics:


,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,img_url,link
count,19311.000000,19311,18336,19311,19311,18605.000000,1.931100e+04,19311.000000,16099.000000,18732.000000,19131,19311
unique,NaN,16214,15194,4851,11127,NaN,NaN,NaN,NaN,NaN,16051,16216
top,NaN,Hakkenden: Touhou Hakken Ibun 2nd Season,No synopsis has been added for this series yet...,['Hentai'],Not available,NaN,NaN,NaN,NaN,NaN,https://cdn.myanimelist.net/images/anime/1071/...,https://myanimelist.net/anime/18055/Hakkenden_...
freq,NaN,3,46,1837,372,NaN,NaN,NaN,NaN,NaN,3,3
mean,19358.904096,NaN,NaN,NaN,NaN,11.460414,3.472609e+04,7720.830304,6866.524194,6.436107,NaN,NaN
std,14271.446515,NaN,NaN,NaN,NaN,47.950386,1.121772e+05,4676.786104,4390.018768,1.007941,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,1.000000,2.500000e+01,1.000000,1.000000,1.250000,NaN,NaN
25%,4833.500000,NaN,NaN,NaN,NaN,1.000000,3.880000e+02,3725.000000,2895.500000,5.770000,NaN,NaN
50%,18327.000000,NaN,NaN,NaN,NaN,2.000000,2.389000e+03,7539.000000,6963.000000,6.410000,NaN,NaN
75%,33896.500000,NaN,NaN,NaN,NaN,12.000000,1.450150e+04,11613.000000,10601.500000,7.150000,NaN,NaN



Missing Values:
uid              0
title            0
synopsis       975
genre            0
aired            0
episodes       706
members          0
popularity       0
ranked        3212
score          579
img_url        180
link             0
dtype: int64

Duplicate Rows:
2943

            Profiles Dataset            

First 5 Rows:


,profile,gender,birthday,favorites_anime,link
0,DesolatePsyche,Male,"Oct 2, 1994","['33352', '25013', '5530', '33674', '1482', '2...",https://myanimelist.net/profile/DesolatePsyche
1,baekbeans,Female,"Nov 10, 2000","['11061', '31964', '853', '20583', '918', '925...",https://myanimelist.net/profile/baekbeans
2,skrn,NaN,NaN,"['918', '2904', '11741', '17074', '23273', '32...",https://myanimelist.net/profile/skrn
3,edgewalker00,Male,Sep 5,"['5680', '849', '2904', '3588', '37349']",https://myanimelist.net/profile/edgewalker00
4,aManOfCulture99,Male,"Oct 30, 1999","['4181', '7791', '9617', '5680', '2167', '4382...",https://myanimelist.net/profile/aManOfCulture99


... (3 outputs left out)

,uid,profile,anime_uid,text,score,scores,link
0,255938,DesolatePsyche,34096,\n \n \n \n ...,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",https://myanimelist.net/reviews.php?id=255938
1,259117,baekbeans,34599,\n \n \n \n ...,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",https://myanimelist.net/reviews.php?id=259117
2,253664,skrn,28891,\n \n \n \n ...,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",https://myanimelist.net/reviews.php?id=253664
3,8254,edgewalker00,2904,\n \n \n \n ...,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",https://myanimelist.net/reviews.php?id=8254
4,291149,aManOfCulture99,4181,\n \n \n \n ...,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",https://myanimelist.net/reviews.php?id=291149



Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192112 entries, 0 to 192111
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        192112 non-null  int64 
 1   profile    192112 non-null  object
 2   anime_uid  192112 non-null  int64 
 3   text       192112 non-null  object
 4   score      192112 non-null  int64 
 5   scores     192112 non-null  object
 6   link       192112 non-null  object
dtypes: int64(3), object(4)
memory usage: 10.3+ MB

Descriptive Statistics:


,uid,profile,anime_uid,text,score,scores,link
count,192112.000000,192112,192112.000000,192112,192112.000000,192112,192112
unique,NaN,47885,NaN,130440,NaN,29837,130519
top,NaN,Sidewinder51,NaN,\n \n \n \n ...,NaN,"{'Overall': '10', 'Story': '10', 'Animation': ...",https://myanimelist.net/reviews.php?id=321183
freq,NaN,762,NaN,8,NaN,13900,4
mean,187648.127525,NaN,15273.300283,NaN,7.570235,NaN,NaN
std,98748.902397,NaN,13480.565379,NaN,2.255167,NaN,NaN
min,1.000000,NaN,1.000000,NaN,0.000000,NaN,NaN
25%,101779.500000,NaN,2167.000000,NaN,6.000000,NaN,NaN
50%,210913.500000,NaN,10793.000000,NaN,8.000000,NaN,NaN
75%,270383.000000,NaN,30205.000000,NaN,9.000000,NaN,NaN



Missing Values:
uid          0
profile      0
anime_uid    0
text         0
score        0
scores       0
link         0
dtype: int64

Duplicate Rows:
61593


# Data Cleaning and Preparation
#### Remove duplicate records and assess missing or inconsistent values across the three sources.

In [5]:
# Duplicate handling

cleaning_summary = []

def clean_duplicates(df, name, key):
    original = len(df)
    exact = int(df.duplicated().sum())
    no_exact = df.drop_duplicates().copy()
    repeated_ids = int(no_exact.duplicated(subset=[key]).sum())
    cleaned = no_exact.drop_duplicates(subset=[key], keep="first").copy()
    cleaning_summary.append({
        "Dataset": name, "Original Rows": original,
        "Exact Duplicates Removed": exact,
        "Additional Duplicate IDs Removed": repeated_ids,
        "Cleaned Rows": len(cleaned)
    })
    return cleaned

animes_clean = clean_duplicates(animes, "Anime", "uid")
profiles_clean = clean_duplicates(profiles, "Profiles", "profile")
reviews_clean = clean_duplicates(reviews, "Reviews", "uid")

cleaning_summary_df = pd.DataFrame(cleaning_summary)
display(cleaning_summary_df)
print("Cleaned shapes:", animes_clean.shape, profiles_clean.shape, reviews_clean.shape)

,Dataset,Original Rows,Exact Duplicates Removed,Additional Duplicate IDs Removed,Cleaned Rows
0,Anime,19311,2943,152,16216
1,Profiles,81727,33825,17,47885
2,Reviews,192112,61593,0,130519


Cleaned shapes: (16216, 12) (47885, 5) (130519, 7)


In [6]:
# Missing Values and Numeric Consistency

for col in ["uid", "episodes", "members", "popularity", "ranked", "score"]:
    animes_clean[col] = pd.to_numeric(animes_clean[col], errors="coerce")
for col in ["uid", "anime_uid", "score"]:
    reviews_clean[col] = pd.to_numeric(reviews_clean[col], errors="coerce")

invalid_overall = int((reviews_clean["score"].notna() & ~reviews_clean["score"].between(1,10)).sum())
reviews_clean["score"] = reviews_clean["score"].where(reviews_clean["score"].between(1,10))
print("Invalid overall review scores:", invalid_overall)

print("Anime missing values after conversion:")
display(animes_clean.isna().sum().to_frame("Missing"))
print("Reviews missing values after conversion:")
display(reviews_clean.isna().sum().to_frame("Missing"))

Invalid overall review scores: 2
Anime missing values after conversion:


,Missing
uid,0
title,0
synopsis,763
genre,0
aired,0
episodes,492
members,0
popularity,0
ranked,1663
score,341


Reviews missing values after conversion:


,Missing
uid,0
profile,0
anime_uid,0
text,0
score,2
scores,0
link,0


# Text Preprocessing

In [7]:
# Download English stopwords
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)

# Load English stopwords
stop_words = set(stopwords.words('english'))

print(f"Total stopwords loaded: {len(stop_words)}")

stop_words.add("etc")

# Preserve negation words
stop_words -= {"no", "not", "nor", "never"}


def preprocess(text):

    if pd.isna(text):
        return ""

    # Convert HTML entities
    text = html.unescape(str(text))

    # Remove anime review rating headers
    text = re.sub(
        r'^\s*(?:more\s+pics\s*)?Overall\s*\d+\s*'
        r'Story\s*\d+\s*Animation\s*\d+\s*'
        r'Sound\s*\d+\s*Character\s*\d+\s*'
        r'Enjoyment\s*\d+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove stopwords
    tokens = [
        word for word in text.split()
        if word not in stop_words
    ]

    return " ".join(tokens)


# Apply preprocessing
reviews_clean["processed_text"] = (
    reviews_clean["text"].apply(preprocess)
)


# Check preprocessing coverage without displaying review text
print("Reviews with processed text:", reviews_clean["processed_text"].ne("").sum())
print("Reviews with empty processed text:", reviews_clean["processed_text"].eq("").sum())


Total stopwords loaded: 198
Reviews with processed text: 130519
Reviews with empty processed text: 0


# Feature Engineering
#### Create anime-level metrics from source attributes, review aggregates, release information, and profile favorites.

In [8]:
# Function to convert genre strings into lists
def parse_genres(value):
    if pd.isna(value) or not str(value).strip():
        return []

    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

In [9]:
# Create the anime-level feature table
anime_features = animes_clean.copy()

# Review features: one row per anime ID
review_summary = (
    reviews_clean.groupby("anime_uid", dropna=True)
    .agg(review_count=("uid", "count"), avg_review_score=("score", "mean"))
    .reset_index()
)

# Popularity index: lower original rank is better
min_rank = anime_features["popularity"].min()
max_rank = anime_features["popularity"].max()
if pd.notna(min_rank) and pd.notna(max_rank) and max_rank > min_rank:
    anime_features["popularity_index"] = (
        (max_rank - anime_features["popularity"]) / (max_rank - min_rank) * 100
    )
else:
    anime_features["popularity_index"] = np.nan

# Convert genre strings into lists
anime_features["genre_list"] = (
    anime_features["genre"].apply(parse_genres)
)

# Genre count
anime_features["genre_count"] = (
    anime_features["genre_list"]
    .apply(lambda genres: len(set(genres)))
)

# Encode genres into binary columns
mlb = MultiLabelBinarizer()

genre_df = pd.DataFrame(
    mlb.fit_transform(anime_features["genre_list"]),
    columns=mlb.classes_,
    index=anime_features.index
)

# Add prefix to genre columns
genre_df = genre_df.add_prefix("genre_")

# Add encoded genres to anime_features
anime_features = pd.concat(
    [anime_features, genre_df],
    axis=1
)

# Remove temporary list column
anime_features.drop(columns=["genre_list"], inplace=True)

# Verify encoding
print("Number of unique genres:", len(genre_df.columns))


# Heuristic franchise flag: title mentions sequels or later installments
franchise_pattern = r"\b(?:season\s*\d+|\d+(?:st|nd|rd|th)\s+season|part\s*\d+|sequel)\b"
anime_features["franchise_indicator"] = (
    anime_features["title"].fillna("")
    .str.contains(franchise_pattern, case=False, regex=True)
    .astype(int)
)

# Episode count is already present: numeric conversion was performed above.

# Merge review aggregates into anime; preserve all anime rows
anime_features = anime_features.merge(
    review_summary, left_on="uid", right_on="anime_uid",
    how="left", validate="one_to_one"
).drop(columns=["anime_uid"])
anime_features["review_count"] = anime_features["review_count"].fillna(0).astype(int)
# avg_review_score remains NaN when there are no matching reviews.

assert anime_features["uid"].is_unique
assert len(anime_features) == len(animes_clean)


Number of unique genres: 43


In [10]:
# =====================================================
# RELEASE YEAR AND ANIME AGE FEATURE ENGINEERING
# =====================================================
# Extract release year and anime age


# Extract the first four-digit year from aired
release_source = animes_clean[["uid", "aired"]].copy()
release_source["release_year"] = pd.to_numeric(
    release_source["aired"].astype(str).str.extract(
        r"\b((?:19|20)\d{2})\b",
        expand=False
    ),
    errors="coerce"
).astype("Int64")

# Calculate anime age using 2026
release_source["anime_age"] = (
    2026 - release_source["release_year"]
).astype("Int64")

# Group anime by release decade
release_source["release_decade"] = (
    (release_source["release_year"] // 10) * 10
).astype("Int64")

# Add features to the anime feature table using anime UID
release_features = release_source[
    ["uid", "release_year", "anime_age", "release_decade"]
].drop_duplicates(subset="uid")

# Remove existing versions if rerunning this cell
anime_features = anime_features.drop(
    columns=["release_year", "anime_age", "release_decade"],
    errors="ignore"
)

anime_features = anime_features.merge(
    release_features,
    on="uid",
    how="left",
    validate="one_to_one"
)


print("Missing release years:",
      anime_features["release_year"].isna().sum())


assert anime_features["uid"].is_unique
assert len(anime_features) == len(animes_clean)


Missing release years: 273


### Content Suitability and Adult Content Identification

In [11]:
# ---- Adult content identification ----

genre_text = anime_features["genre"].fillna("").astype(str)

# Flag explicit and adult-oriented content
anime_features["adult_content_flag"] = (
    genre_text.str.contains(
        r"\b(?:Hentai|Erotica)\b",
        case=False,
        regex=True
    ).astype(int)
)

# Flag sexually suggestive content
anime_features["suggestive_content_flag"] = (
    genre_text.str.contains(
        r"\bEcchi\b",
        case=False,
        regex=True
    ).astype(int)
)

# Classify content for licensing review
anime_features["content_category"] = np.select(
    [
        anime_features["adult_content_flag"] == 1,
        anime_features["suggestive_content_flag"] == 1,
        genre_text.str.strip().isin(["", "[]"])
    ],
    [
        "Adult Content",
        "Suggestive Content",
        "Unknown"
    ],
    default="No Adult Genre Identified"
)

# Display content distribution
print(anime_features["content_category"].value_counts())


assert anime_features["uid"].is_unique
assert len(anime_features) == len(animes_clean)


content_category
No Adult Genre Identified    14107
Adult Content                 1294
Suggestive Content             740
Unknown                         75
Name: count, dtype: int64


In [12]:
# Profile favorites feature

# favorites_anime contains strings such as "['33352', '25013']"
def parse_favorites(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []

profiles_fav = profiles_clean[["profile", "favorites_anime"]].copy()
profiles_fav["favorites_anime"] = profiles_fav["favorites_anime"].apply(parse_favorites)
favorites = profiles_fav.explode("favorites_anime")
favorites["favorites_anime"] = pd.to_numeric(favorites["favorites_anime"], errors="coerce")
favorites = favorites.dropna(subset=["favorites_anime", "profile"])
favorites = favorites.drop_duplicates(subset=["profile", "favorites_anime"])

favorites_summary = (
    favorites.groupby("favorites_anime")["profile"]
    .nunique().reset_index(name="favorites_count")
)

anime_features = anime_features.merge(
    favorites_summary, left_on="uid", right_on="favorites_anime",
    how="left", validate="one_to_one"
).drop(columns=["favorites_anime"])
anime_features["favorites_count"] = anime_features["favorites_count"].fillna(0).astype(int)

print("Total matched profile–anime favorites:", anime_features["favorites_count"].sum())

assert anime_features["uid"].is_unique
assert len(anime_features) == len(animes_clean)


Total matched profile–anime favorites: 216297


In [13]:
# Parse score dictionaries
def parse_scores(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError):
        return {}

# Expand scores into separate columns
parsed_scores = reviews_clean['scores'].apply(parse_scores)

score_cols = pd.json_normalize(parsed_scores.tolist())
score_cols.index = reviews_clean.index

# Convert to numeric and rename columns
score_cols = score_cols.apply(pd.to_numeric, errors='coerce')
score_cols = score_cols.add_prefix('review_')

# Add score columns to reviews_clean
reviews_clean = pd.concat([
    reviews_clean.drop(columns=score_cols.columns, errors='ignore'),
    score_cols
], axis=1)

# Check and clean component ratings
components = ["Overall", "Story", "Animation", "Sound", "Character", "Enjoyment"]
score_columns = [f"review_{name}" for name in components]
ratings = reviews_clean[score_columns]
valid = ratings.ge(1) & ratings.le(10)
review_cleaning_summary = pd.DataFrame({
    "Zero ratings": ratings.eq(0).sum(),
    "Other invalid ratings": (ratings.notna() & ~valid & ratings.ne(0)).sum(),
    "Missing before cleaning": ratings.isna().sum()
})
reviews_clean[score_columns] = ratings.where(valid)
display(review_cleaning_summary)


,Zero ratings,Other invalid ratings,Missing before cleaning
review_Overall,1,1,0
review_Story,5910,1,0
review_Animation,5981,1,0
review_Sound,6283,1,0
review_Character,6058,1,0
review_Enjoyment,6350,1,0


In [14]:
# Aggregate valid review ratings and counts per anime
ratings_group = reviews_clean.groupby("anime_uid")[score_columns]
averages = ratings_group.mean().add_prefix("avg_")
counts = ratings_group.count().add_prefix("count_")
review_components = averages.join(counts)
anime_features = anime_features.merge(
    review_components, left_on="uid", right_index=True,
    how="left", validate="one_to_one"
)
anime_features[counts.columns] = anime_features[counts.columns].fillna(0).astype(int)


### Final feature-engineered dataset
All anime-level features have now been incorporated. Verify the schema and primary key before assessing data quality.

In [15]:
final_dataset = anime_features.copy()

assert final_dataset.shape[1] == 79, "Unexpected feature count"
assert final_dataset["uid"].notna().all()
assert final_dataset["uid"].is_unique
assert len(final_dataset) == len(animes_clean)

print("Final dataset shape:", final_dataset.shape)
print("Duplicate anime IDs:", final_dataset["uid"].duplicated().sum())
display(final_dataset.head())

Final dataset shape: (16216, 79)
Duplicate anime IDs: 0


,uid,title,synopsis,genre,aired,episodes,members,popularity,ranked,score,...,avg_review_Animation,avg_review_Sound,avg_review_Character,avg_review_Enjoyment,count_review_Overall,count_review_Story,count_review_Animation,count_review_Sound,count_review_Character,count_review_Enjoyment
0,28891,Haikyuu!! Second Season,Following their participation at the Inter-Hig...,"['Comedy', 'Sports', 'Drama', 'School', 'Shoun...","Oct 4, 2015 to Mar 27, 2016",25.0,489888,141,25.0,8.82,...,8.625000,8.916667,8.851064,9.000000,52,48,48,48,47,48
1,23273,Shigatsu wa Kimi no Uso,Music accompanies the path of the human metron...,"['Drama', 'Music', 'Romance', 'School', 'Shoun...","Oct 10, 2014 to Mar 20, 2015",22.0,995473,28,24.0,8.83,...,8.400000,8.700000,6.000000,6.150000,20,20,20,20,20,20
2,34599,Made in Abyss,The Abyss—a gaping chasm stretching down into ...,"['Sci-Fi', 'Adventure', 'Mystery', 'Drama', 'F...","Jul 7, 2017 to Sep 29, 2017",13.0,581663,98,23.0,8.83,...,8.969178,8.965753,8.167808,8.657534,305,293,292,292,292,292
3,5114,Fullmetal Alchemist: Brotherhood,"""In order for something to be obtained, someth...","['Action', 'Military', 'Adventure', 'Comedy', ...","Apr 5, 2009 to Jul 4, 2010",64.0,1615084,4,1.0,9.23,...,8.974441,9.023923,9.181529,9.287081,637,627,626,627,628,627
4,31758,Kizumonogatari III: Reiketsu-hen,After helping revive the legendary vampire Kis...,"['Action', 'Mystery', 'Supernatural', 'Vampire']","Jan 6, 2017",1.0,214621,502,22.0,8.83,...,9.000000,8.769231,8.192308,8.500000,28,26,26,26,26,26


## Data Quality Assessment

In [16]:
df = final_dataset
missing_counts = df.isna().sum()
placeholders = ["", "not available", "unknown", "n/a", "none", "null"]
for col in df.select_dtypes(include=["object", "string"]):
    if col != "content_category":
        missing_counts[col] += df[col].astype("string").str.strip().str.lower().isin(placeholders).sum()

completeness_report = pd.DataFrame({"Missing Count": missing_counts,
    "Missing %": (missing_counts / len(df) * 100).round(2)})
completeness_report["Completeness %"] = 100 - completeness_report["Missing %"]
completeness_report.index.name = "Column"

genre_cols = [col for col in df if col.startswith("genre_") and col != "genre_count"]
avg_cols = [f"avg_review_{name}" for name in components]
count_cols = [f"count_review_{name}" for name in components]
validity = {}
for col, low, high in [("score", 0, 10), ("popularity_index", 0, 100),
                       ("avg_review_score", 1, 10)] + [(col, 1, 10) for col in avg_cols]:
    validity[col + " outside range"] = (df[col].notna() & ~df[col].between(low, high)).sum()
for col in ["episodes", "members", "review_count", "favorites_count", "anime_age"]:
    validity[col + " negative"] = df[col].lt(0).sum()
validity["Invalid popularity rank"] = df["popularity"].lt(1).sum()
for col in count_cols:
    validity[col + " invalid"] = (df[col].isna() | df[col].lt(0) | df[col].mod(1).ne(0)).sum()
flags = genre_cols + ["adult_content_flag", "suggestive_content_flag", "franchise_indicator"]
validity["Invalid binary flags"] = (~df[flags].isin([0, 1])).sum().sum()
categories = ["Adult Content", "Suggestive Content", "Unknown", "No Adult Genre Identified"]
validity["Invalid content category"] = (~df.content_category.isin(categories)).sum()
validity_report = pd.Series(validity, name="Invalid Records").rename_axis("Validation Check").reset_index()

known = df.release_year.notna()
consistency = {
    "Genre count mismatch": df[genre_cols].sum(axis=1).ne(df.genre_count).sum(),
    "Release decade mismatch": (known & (df.release_decade.isna() | df.release_decade.ne(df.release_year // 10 * 10))).sum(),
    "Anime age mismatch": (known & (df.anime_age.isna() | df.anime_age.ne(2026 - df.release_year))).sum(),
    "Derived dates without release year": (~known & (df.anime_age.notna() | df.release_decade.notna())).sum(),
    "Review score without reviews": (df.review_count.eq(0) & df.avg_review_score.notna()).sum()
}
for avg, count in zip(avg_cols, count_cols):
    consistency[avg + " / count mismatch"] = ((df[count].eq(0) & df[avg].notna()) |
        (df[count].gt(0) & df[avg].isna()) | df[count].gt(df.review_count)).sum()
consistency_report = pd.Series(consistency, name="Count").rename_axis("Check").reset_index()
review_coverage = pd.DataFrame({"Component": components,
    "Anime With Ratings": df[avg_cols].notna().sum().values,
    "Missing Ratings": df[avg_cols].isna().sum().values,
    "Mean Score": df[avg_cols].mean().round(2).values,
    "Total Valid Ratings": df[count_cols].sum().values})
review_coverage["Coverage %"] = (review_coverage["Anime With Ratings"] / len(df) * 100).round(2)
quality_summary = pd.Series({
    "Total records": len(df), "Total columns": df.shape[1],
    "Overall completeness (%)": round(100 * (1 - missing_counts.sum() / df.size), 2),
    "Duplicate rows": df.duplicated().sum(), "Duplicate anime IDs": df.uid.duplicated().sum(),
    "Missing anime IDs": df.uid.isna().sum(),
    "Invalid numeric/encoded values": validity_report["Invalid Records"].sum(),
    "Consistency issues": consistency_report["Count"].sum(),
    "Binary genre columns": len(genre_cols)
}, name="Result").rename_axis("Metric").reset_index()
for report in [quality_summary, completeness_report, validity_report, consistency_report, review_coverage]:
    display(report)


,Metric,Result
0,Total records,16216.00
1,Total columns,79.00
2,Overall completeness (%),95.14
3,Duplicate rows,0.00
4,Duplicate anime IDs,0.00
5,Missing anime IDs,0.00
6,Invalid numeric/encoded values,0.00
7,Consistency issues,0.00
8,Binary genre columns,43.00


,Missing Count,Missing %,Completeness %
Column,,,
uid,0,0.00,100.00
title,0,0.00,100.00
synopsis,770,4.75,95.25
genre,0,0.00,100.00
aired,273,1.68,98.32
...,...,...,...
count_review_Story,0,0.00,100.00
count_review_Animation,0,0.00,100.00
count_review_Sound,0,0.00,100.00


,Validation Check,Invalid Records
0,score outside range,0
1,popularity_index outside range,0
2,avg_review_score outside range,0
3,avg_review_Overall outside range,0
4,avg_review_Story outside range,0
5,avg_review_Animation outside range,0
6,avg_review_Sound outside range,0
7,avg_review_Character outside range,0
8,avg_review_Enjoyment outside range,0
9,episodes negative,0


,Check,Count
0,Genre count mismatch,0
1,Release decade mismatch,0
2,Anime age mismatch,0
3,Derived dates without release year,0
4,Review score without reviews,0
5,avg_review_Overall / count mismatch,0
6,avg_review_Story / count mismatch,0
7,avg_review_Animation / count mismatch,0
8,avg_review_Sound / count mismatch,0
9,avg_review_Character / count mismatch,0


,Component,Anime With Ratings,Missing Ratings,Mean Score,Total Valid Ratings,Coverage %
0,Overall,8113,8103,6.77,130517,50.03
1,Story,7921,8295,6.37,124608,48.85
2,Animation,7932,8284,7.15,124537,48.91
3,Sound,7921,8295,7.07,124235,48.85
4,Character,7913,8303,6.73,124460,48.80
5,Enjoyment,7922,8294,7.04,124168,48.85


# Data Dictionary
Document each feature, its type and missing-value counts. Genre and review component descriptions are generated automatically.

In [17]:
definitions = {
    'uid': 'Unique anime identifier',
    'title': 'Anime title',
    'synopsis': 'Anime description',
    'genre': 'Anime genre labels',
    'aired': 'Release or broadcast information',
    'episodes': 'Number of episodes',
    'members': 'Number of anime community members',
    'popularity': 'Popularity ranking; lower is better',
    'ranked': 'Overall anime ranking; lower is better',
    'score': 'Overall anime rating',
    'img_url': 'Anime image URL',
    'link': 'Anime information URL',
    'popularity_index': 'Normalized popularity score (0-100)',
    'genre_count': 'Number of unique genres assigned to each anime',
    'franchise_indicator': 'Possible related-release title flag',
    'review_count': 'Number of reviews for each anime',
    'adult_content_flag': '1 if genre text contains Hentai or Erotica; otherwise 0.',
    'suggestive_content_flag': '1 if genre text contains Ecchi; otherwise 0.',
    'content_category': 'Genre-based category; adult takes precedence over suggestive, then unknown for blank genre lists. Not an age rating.',
    'favorites_count': 'Number of profiles favoriting the anime',
    'release_year': 'First release year extracted from the aired column',
    'anime_age': 'Years since first release, calculated as of 2026',
    'release_decade': 'Decade of first release, used for historical trend analysis',
    'avg_review_score': 'Mean valid overall review score (1–10); missing if none.',
}
for col in genre_cols:
    definitions[col] = f"1 if tagged {col[6:]}; otherwise 0."
for name in components:
    definitions[f"avg_review_{name}"] = f"Mean valid {name} rating (1–10); missing if none."
    definitions[f"count_review_{name}"] = f"Number of valid {name} ratings; 0 if none."

assert set(df.columns) == set(definitions), "Dictionary does not match dataset schema"
data_dictionary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Description": df.columns.map(definitions),
    "Missing Values": df.isna().sum().values,
    "Missing Including Placeholders": missing_counts.values
})
display(data_dictionary)


,Column,Data Type,Description,Missing Values,Missing Including Placeholders
0,uid,int64,Unique anime identifier,0,0
1,title,object,Anime title,0,0
2,synopsis,object,Anime description,763,770
3,genre,object,Anime genre labels,0,0
4,aired,object,Release or broadcast information,0,273
...,...,...,...,...,...
74,count_review_Story,int64,Number of valid Story ratings; 0 if none.,0,0
75,count_review_Animation,int64,Number of valid Animation ratings; 0 if none.,0,0
76,count_review_Sound,int64,Number of valid Sound ratings; 0 if none.,0,0
77,count_review_Character,int64,Number of valid Character ratings; 0 if none.,0,0


# Cleaning Summary
Record the cleaning and coverage statistics for the separate engineering report.

In [18]:
cleaning_metrics = pd.Series({
    "Invalid overall review scores removed": invalid_overall,
    "Anime with reviews": final_dataset.review_count.gt(0).sum(),
    "Anime without reviews": final_dataset.review_count.eq(0).sum(),
    "Missing release years": final_dataset.release_year.isna().sum(),
    "Matched favorites": final_dataset.favorites_count.sum(),
    "Earliest release year": final_dataset.release_year.min(),
    "Latest release year": final_dataset.release_year.max()
}, name="Result").rename_axis("Metric").reset_index()
content_summary = final_dataset.content_category.value_counts().rename_axis("Category").reset_index(name="Titles")
display(cleaning_summary_df, cleaning_metrics, content_summary)


,Dataset,Original Rows,Exact Duplicates Removed,Additional Duplicate IDs Removed,Cleaned Rows
0,Anime,19311,2943,152,16216
1,Profiles,81727,33825,17,47885
2,Reviews,192112,61593,0,130519


,Metric,Result
0,Invalid overall review scores removed,2
1,Anime with reviews,8113
2,Anime without reviews,8103
3,Missing release years,273
4,Matched favorites,216297
5,Earliest release year,1917
6,Latest release year,2021


,Category,Titles
0,No Adult Genre Identified,14107
1,Adult Content,1294
2,Suggestive Content,740
3,Unknown,75


# Export Final Deliverables
Export the analysis-ready dataset and supporting CSV tables. The formal Word documents are prepared separately.

In [19]:
OUTPUT_DIR = Path("Anime_Project_Output_Final")
OUTPUT_DIR.mkdir(exist_ok=True)
exports = {
    "final_anime_dataset": final_dataset,
    "anime_clean": animes_clean,
    "profiles_clean": profiles_clean,
    "reviews_clean": reviews_clean
}
for name, table in exports.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
print(f"Exported {len(exports)} files to {OUTPUT_DIR.resolve()}")


Exported 4 files to C:\Users\atosh\ITDS 2373_LABS\CAPSTONE PROJECT\Anime_Project_Output_Final\Anime_Project_Output_Final
